# NLP For ML

In [3]:
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
# Raw data --> Corpus

## 1. Tokenization

In [4]:
# Why not just use .split()
text.split()

['Apple',
 'Inc.',
 'was',
 'founded',
 'by',
 'Steve',
 'Jobs',
 'in',
 'Cupertino,',
 'California.']

### Problem with `.split()`

`.split()` splits only on **whitespace** - it has no understanding of language, so punctuation stays glued to words:

- `"Cupertino,"` → should be `"Cupertino"` + `","`
- `"California."` → should be `"California"` + `"."`

This breaks everything downstream - `"California."` and `"California"` would be treated as **two different words** by the model, even though they're the same.

NLTK's tokenizer understands language rules - it knows `.` after `Inc` is an abbreviation, not a sentence end, and separates punctuation correctly.

### Install NLTK

In [ ]:
# !pip install nltk

In [ ]:
import nltk

Download pre-trained tokenizer models - `punkt` and `punkt_tab` teach NLTK where sentences and words begin/end, without them `sent_tokenize` and `word_tokenize` won't work.

In [ ]:
# Downloads the pre-trained sentence/word tokenizer models used below.
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\scl\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\scl\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

### Sentence Tokenization

In [ ]:
from nltk.tokenize import sent_tokenize

document = (
    "Apple Inc. was founded by Steve Jobs in Cupertino, California. "
    "The company is now planning to open a new office in New York City. "
    "Their engineers are running experiments on advanced machine learning models, "
    "and the results have been better than expected."
)

In [ ]:
document

'Apple Inc. was founded by Steve Jobs in Cupertino, California. The company is now planning to open a new office in New York City. Their engineers are running experiments on advanced machine learning models, and the results have been better than expected.'

In [ ]:
sentences = sent_tokenize(document)

In [ ]:
for i, s in enumerate(sentences):
    print(f"{i} : {s}")

0 : Apple Inc. was founded by Steve Jobs in Cupertino, California.
1 : The company is now planning to open a new office in New York City.
2 : Their engineers are running experiments on advanced machine learning models, and the results have been better than expected.


### Why not just use Sentence Tokenization?

Sentence tokenizer splits text into **sentences** - each sentence is one token. That's useful for understanding document structure, but ML models don't work with whole sentences directly.

Word tokenizer splits into **individual words** - because ML models need numbers, and you convert **each word** into a number (via vocabulary lookup, embeddings, etc.).

**The real flow is:**
```
Document → [Sentence Tokenize] → Sentences → [Word Tokenize] → Words → [Numbers for ML]
```

Sentence tokenize is a **stepping stone** - you often word-tokenize sentence by sentence so word boundaries don't bleed across sentences. Word tokens are the actual unit ML learns from.

### Word Tokenization

In [ ]:
from nltk.tokenize import word_tokenize

first_sentence = sentences[0]

print("Naive .split()", first_sentence.split())
print("NTLK word_tokenize", word_tokenize(first_sentence))

Naive .split() ['Apple', 'Inc.', 'was', 'founded', 'by', 'Steve', 'Jobs', 'in', 'Cupertino,', 'California.']
NTLK word_tokenize ['Apple', 'Inc.', 'was', 'founded', 'by', 'Steve', 'Jobs', 'in', 'Cupertino', ',', 'California', '.']


In [ ]:
# tokenize full document

all_token = word_tokenize(document)
print(f"Total Tokens : {len(all_token)}")
print(f"{all_token}")

Total Tokens : 47
['Apple', 'Inc.', 'was', 'founded', 'by', 'Steve', 'Jobs', 'in', 'Cupertino', ',', 'California', '.', 'The', 'company', 'is', 'now', 'planning', 'to', 'open', 'a', 'new', 'office', 'in', 'New', 'York', 'City', '.', 'Their', 'engineers', 'are', 'running', 'experiments', 'on', 'advanced', 'machine', 'learning', 'models', ',', 'and', 'the', 'results', 'have', 'been', 'better', 'than', 'expected', '.']


# 2. Text Normalization

### Why normalize at all?

The same word appears in many forms, and ML models treat each form as a completely different word:

- `"running"`, `"runs"`, `"ran"` → all mean **run**, but the model sees 3 different tokens
- `"studies"`, `"studying"`, `"studied"` → all mean **study**, but again 3 different tokens

This causes two problems:

1. **Vocabulary explodes** - the model has to learn thousands of word variants instead of the core words
2. **Loses meaning connection** - the model won't know `"ran"` and `"running"` are related

Stemming/Lemmatization collapse all variants to one base form → smaller vocabulary, better generalization.

problem-1 (lemma --> Base form)

- running
- runs
- ran

--> [Stemming (fast, not accurate), Lemmatization(slow, accurate)]

problem-2 (Stop words)

- the
- is
- a

### Stemming

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
# rules-basic -> "if a word ends in -ing it will remove that part"

words = ["Running", "runs", "ran", "easily", "fairness", "studies", "better", "geese"]

for w in words:
    print(f"{w} --> {stemmer.stem(w)}")


Running --> run
runs --> run
ran --> ran
easily --> easili
fairness --> fair
studies --> studi
better --> better
geese --> gees


### Lemmatization

In [ ]:
nltk.download("wordnet")
# the dictionary/database the lemmatizer looks words up in

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\scl\AppData\Roaming\nltk_data...


True

In [ ]:
from nltk.stem import WordNetLemmatizer

In [ ]:
lemmatizer = WordNetLemmatizer()

for w in words:
    print(f"{w} --> {lemmatizer.lemmatize(w)}")

Running --> Running
runs --> run
ran --> ran
easily --> easily
fairness --> fairness
studies --> study
better --> better
geese --> goose


In [ ]:
# lemmatizer defaults to assuming every word is a noun.
# solution : tell the lemmatizer to use verb

verbs_to_check = ["Running", "runs", "ran", "talking", "studies"]

print("Default (Assumes noun) : \n")
for w in verbs_to_check:
    print(f"{w} --> {lemmatizer.lemmatize(w)}")

print("==="*10)

# A verb is a word that shows an action, an occurrence, or a state of being
print("Explicitly told pos(parts of speech) to be verb pos='v' : \n")
for w in verbs_to_check:
    print(f"{w} --> {lemmatizer.lemmatize(w, pos='v')}")

Default (Assumes noun) : 

Running --> Running
runs --> run
ran --> ran
talking --> talking
studies --> study
!? --> !?
Explicitly told pos(parts of speech) to be verb pos='v' : 

Running --> Running
runs --> run
ran --> run
talking --> talk
studies --> study
!? --> !?


In [ ]:
# Problem : statement has Verb, noun and adjective

Fast
- stemming (rules)
Slow
- Lemmatization (dict lookup)

Accuracy
- Lemmatization

Stemming is good for :
- Quick prototypes
- search/retrival system
- huge corpora where speed matters

Lemmatization is good for :
- anything where output readability matters
- where accuracy on irregular words matters

### Stopwords removal

In [ ]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\scl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english"))
print(f"Total English stopwords: {len(english_stopwords)}")
print(sorted(english_stopwords))

Total English stopwords: 198
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", "he's", 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', '

In [ ]:
sentence = "The engineers are running experiments on advanced machine learning models"

tokens = word_tokenize(sentence)

filtered_tokens = [t for t in tokens if t.lower() not in english_stopwords]

print("Before:", tokens)
print("After: ", filtered_tokens)

Before: ['The', 'engineers', 'are', 'running', 'experiments', 'on', 'advanced', 'machine', 'learning', 'models']
After:  ['engineers', 'running', 'experiments', 'advanced', 'machine', 'learning', 'models']


### Putting it all together

In [ ]:
def preprocess(text):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    # tokens = [t.lower() for t in token if t.isalpha()] # --> drop the punctuation
    tokens = [t for t in tokens if t not in english_stopwords]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

In [ ]:
sample = "The engineers were quickly running several experiments on the advanced learning models."
print(preprocess(sample))

['engineer', 'quickly', 'running', 'several', 'experiment', 'advanced', 'learning', 'model', '.']


# 3. Parts-Of-Speech Tagging and Named Entity Recognition

### POS Tagging

Labelling each token with its grammatical role : [noun, verb, adjective, preposition, and so on]

"a good run"
"i run every data"

NLTK uses the Penn Treebank tag set, a fixed vocab of short tag code

| Tag | Meaning | Example |
|---|---|---|
| `NN` | Noun, singular | `dog` |
| `NNS` | Noun, plural | `dogs` |
| `NNP` | Proper noun, singular | `Apple`, `Steve` |
| `VB` | Verb, base form | `run` |
| `VBD` | Verb, past tense | `ran` |
| `VBG` | Verb, gerund/present participle | `running` |
| `VBZ` | Verb, 3rd person singular present | `runs` |
| `JJ` | Adjective | `advanced` |
| `RB` | Adverb | `quickly` |
| `IN` | Preposition/subordinating conjunction | `in`, `by`, `on` |
| `DT` | Determiner | `the`, `a` |
| `,` / `.` | Literal punctuation | `,` `.` |

- N* -> Noun
- V* -> Verb
- J* -> Adjective
- R* -> Adverb

### POS Tag

In [ ]:
nltk.download("averaged_perceptron_tagger_eng")
#It's the pre-trained model that pos_tag() uses - without downloading it, pos_tag() won't work

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\scl\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
from nltk import pos_tag
from nltk.tokenize import word_tokenize

In [ ]:
sentence = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
tokens = word_tokenize(sentence)

tagged = pos_tag(tokens)
for word, tag in tagged:
    print(f"{word} : {tag}")

Apple : NNP
Inc. : NNP
was : VBD
founded : VBN
by : IN
Steve : NNP
Jobs : NNP
in : IN
Cupertino : NNP
, : ,
California : NNP
. : .


In [ ]:
# - N* -> Noun
# - V* -> Verb
# - J* -> Adjective
# - R* -> Adverb
from nltk.corpus import wordnet

def penn_towordnet_pos(penn_tag):
    if penn_tag.startswith("J"):
        return wordnet.ADJ
    elif penn_tag.startswith("V"):
        return wordnet.VERB
        # return 'v'
    elif penn_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def preprocess_v2(text):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    # tokens = [t.lower() for t in token if t.isalpha()] # --> drop the punctuation
    tokens = [t for t in tokens if t not in english_stopwords]
    tagged = pos_tag(tokens)
    # pos=wordnet.VERB --> pos='v'
    return [lemmatizer.lemmatize(word, pos=penn_towordnet_pos(tag)) for word, tag in tagged]


In [ ]:
sample = "The engineers were quickly running several experiments on the advanced learning models."
print("Lemma Without POS TAG", preprocess(sample))
print("Lemma With POS TAG", preprocess_v2(sample))

Lemma Without POS TAG ['engineer', 'quickly', 'running', 'several', 'experiment', 'advanced', 'learning', 'model', '.']
Lemma With POS TAG ['engineer', 'quickly', 'run', 'several', 'experiment', 'advance', 'learning', 'model', '.']


In [ ]:
tokens = word_tokenize(sample)
tokens = [t.lower() for t in tokens]
# tokens = [t.lower() for t in token if t.isalpha()] # --> drop the punctuation
tokens = [t for t in tokens if t not in english_stopwords]
tagged = pos_tag(tokens)
[(lemmatizer.lemmatize(word, pos=penn_towordnet_pos(tag)), tag) for word, tag in tagged]

[('engineer', 'NNS'),
 ('quickly', 'RB'),
 ('run', 'VBG'),
 ('several', 'JJ'),
 ('experiment', 'NNS'),
 ('advance', 'VBD'),
 ('learning', 'JJ'),
 ('model', 'NNS'),
 ('.', '.')]

### NER : Named Entity Recognition

In [ ]:
nltk.download("maxent_ne_chunker_tab")
nltk.download("words") # a large dictionary of known English words the chunker consults

# These are the two resources ne_chunk() needs to run.



[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     C:\Users\scl\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping chunkers\maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\scl\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\words.zip.


True

`ne_chunk()` needs POS tags as input, not plain tokens - the pipeline is strictly `Text → word_tokenize → pos_tag → ne_chunk`

In [ ]:
from nltk import ne_chunk

sentence = "Apple Inc. was founded by Steve Jobs in Cupertino, California."
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)
entity_tree = ne_chunk(tagged) # ne_chunk takes POS-tagged tokens
print(entity_tree)

(S
  (PERSON Apple/NNP)
  (ORGANIZATION Inc./NNP)
  was/VBD
  founded/VBN
  by/IN
  (PERSON Steve/NNP Jobs/NNP)
  in/IN
  (GPE Cupertino/NNP)
  ,/,
  (GPE California/NNP)
  ./.)


In [ ]:
# GPE -> geo Political entity / places
# documented weakes of this specific model :accuracy
# We can use library like SpaCy or fine-tunes transformer

# EXTRAS : NER With SPACY

Same NER task, but spaCy does every step internally.

`en_core_web_sm` is a **pre-trained pipeline** - one bundle containing the tokenizer, POS tagger, parser, lemmatizer and NER model:
- **en** → English, **core** → general purpose, **web** → trained on web text, **sm** → small (~12 MB)

So `nlp(sentence)` runs `tokenize → pos_tag → ner` in a single call, and `doc.ents` gives back the entities directly.

| | NLTK | spaCy |
|---|---|---|
| Resources to download | 6 (`punkt`, `wordnet`, `tagger`, `chunker`, ...) | 1 (`en_core_web_sm`) |
| Steps you write | 3 | 1 |
| Output | a tree you must walk | a clean list (`doc.ents`) |
| `Apple Inc.` | ❌ `PERSON` + `ORGANIZATION` | ✅ one `ORG` |

NLTK shows you every step, spaCy hides them - that's why we learn NLTK first.

In [ ]:
# !pip install spacy

In [ ]:
import spacy

In [ ]:
# !python -m spacy download en_core_web_sm

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
sentence = "Apple Inc. was founded by Steve Jobs in Cupertino, California."

doc = nlp(sentence)

for ent in doc.ents:
    print(f"{ent.text:<15} -> {ent.label_}")

Apple Inc.      -> ORG
Steve Jobs      -> PERSON
Cupertino       -> GPE
California      -> GPE


# Assignment : Convert the below pipeline with spacy

```bash
# - N* -> Noun
# - V* -> Verb
# - J* -> Adjective
# - R* -> Adverb
from nltk.corpus import wordnet

def penn_towordnet_pos(penn_tag):
    if penn_tag.startswith("J"):
        return wordnet.ADJ
    elif penn_tag.startswith("V"):
        return wordnet.VERB
        # return 'v'
    elif penn_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def preprocess_v2(text):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    # tokens = [t.lower() for t in token if t.isalpha()] # --> drop the punctuation
    tokens = [t for t in tokens if t not in english_stopwords]
    tagged = pos_tag(tokens)
    # pos=wordnet.VERB --> pos='v'
    return [lemmatizer.lemmatize(word, pos=penn_towordnet_pos(tag)) for word, tag in tagged]
```